# A simple MODFLOW-USG example

Generate a coarse Voronoi mesh, export a matching DISU/GSF pair, and simulate steady confined flow between two fixed-head sides. The other sides have no flow. There are no wells, barriers, recharge or transient stresses.

This uses **MODFLOW-USG**, not MODFLOW 6. Both engines have DISU packages, but their input arrays and connection conventions differ. `to_mfusg_disu()` and `to_mfusgdisu()` target USG only.

From the repository root, install the editable package and example dependencies in the environment used by the notebook kernel:

```sh
python -m pip install -e ".[examples]"
```

Separately, create a binary directory and run `get-modflow /path/to/bin --subset mfusg`. Add that directory to PATH, or set `MFUSG_EXE` to the executable's full path before running this notebook. The notebook never downloads executables. For example, on Windows PowerShell:

```powershell
$env:MFUSG_EXE = "C:\modflow\bin\mfusg.exe"
```

For optional barrier source layouts, repairs and side tracking, see [the MF6 notebook](basic_example_modflow.ipynb).

In [ ]:
import os
from pathlib import Path
import shutil
import tempfile

import numpy as np
from shapely.geometry import box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator

# Import the meshing package before FloPy and its plotting backend.
import flopy
import matplotlib.pyplot as plt

exe_request = os.environ.get("MFUSG_EXE", "mfusg")
mfusg_exe = shutil.which(exe_request)
if mfusg_exe is None:
    raise FileNotFoundError(
        f"MODFLOW-USG executable not found: {exe_request!r}. "
        "Install mfusg separately with get-modflow /path/to/bin --subset mfusg, "
        "then set MFUSG_EXE to its full path or add its directory to PATH."
    )

## Define and generate the mesh

The rectangle is 200 × 100 m, with a uniform characteristic mesh size of 20 m. Model cell numbers follow the final grid's row positions, not its GeoDataFrame index or generator IDs. The USG export below validates geometry and reports connection orthogonality separately from the head and flow checks.

In [ ]:
domain = box(0, 0, 200, 100)
cm = ConceptualMesh()
cm.add_polygon(domain, zone_id=1, resolution=20.0)
mg = MeshGenerator(background_lc=20.0, conceptual_mesh=cm)
mg.generate()
vt = VoronoiTessellator(mg, cm, clip_to_boundary=True)
gdf = vt.generate()

fig, ax = plt.subplots(figsize=(8, 4))
vt.plot(ax=ax, column=None)
ax.set_aspect("equal")
ax.set_title("Voronoi mesh")
plt.show()

## Build the MODFLOW-USG model

Use one confined layer, top 10 m, bottom 0 m, and hydraulic conductivity 10 m/day. Heads above the layer top are valid for this confined example. FloPy writes the DISU package together with the other model inputs.

The plain-array route is `vt.to_mfusg_disu(top, botm, nlay=1)`. Its `node_id`, `cell_map` and `connectivity` entries are mapping/diagnostic metadata, not FloPy arguments. The package wrapper below removes those entries automatically. The map records zero-based model nodes, horizontal row positions, original index labels and generator IDs.

Inputs and results go into a temporary directory. To retain them, use the permanent `model_ws` alternative in the next cell. Final cleanup removes only the temporary directory.

In [ ]:
workspace = tempfile.TemporaryDirectory()
model_ws = Path(workspace.name)
# To retain model.disu, model.gsf and results, use this permanent path instead:
# model_ws = Path("mfusg_example").resolve()
# model_ws.mkdir(parents=True, exist_ok=True)

model = flopy.mfusg.MfUsg(
    modelname="model", structured=False,
    model_ws=model_ws, exe_name=mfusg_exe,
)
top, botm, nlay = 10.0, [0.0], 1
grid = vt.to_mfusg_disu(top=top, botm=botm, nlay=nlay)
max_ortho_error = float(grid["connectivity"]["ortho_error"].max())
assert np.isfinite(max_ortho_error) and max_ortho_error < 1e-4
print(f"Maximum connection orthogonality error: {max_ortho_error:.3g} degrees")
mapping = grid["cell_map"]
layer_map = mapping.loc[mapping["layer"] == 0]
assert np.array_equal(layer_map["cell_id"], np.arange(len(gdf)))
assert layer_map["orig_index"].tolist() == gdf.index.tolist()
display(mapping.head())

disu = vt.to_mfusgdisu(
    model, top=top, botm=botm, nlay=nlay,
    nper=1, perlen=1.0, nstp=1, steady=True, itmuni=4, lenuni=2,
)
# Use identical geometry/elevations for both files. GSF contains no stresses.
gsf_path = vt.to_gsf(model_ws / "model.gsf", top=top, botm=botm, nlay=nlay)

## Boundary conditions, properties and solver

Fix the heads at **numerical cell centres** on x = 0 m and x = 200 m to 20 m and 10 m respectively. Selecting every polygon that touches a side can include centres inside the rectangle, changing the effective distance between prescribed heads.

Negative `ibound` values mark constant-head cells in MODFLOW-USG. Unassigned outer faces have no flow. Interior initial heads deliberately differ from the analytical solution, so the solver must recover it.

In [ ]:
x = gdf["x"].to_numpy()
left = np.flatnonzero(np.isclose(x, 0.0, atol=1e-7, rtol=0))
right = np.flatnonzero(np.isclose(x, 200.0, atol=1e-7, rtol=0))
assert len(left) and len(right), "Expected generators on both constant-head sides."

conductivity = 10.0  # m/day
ibound = np.ones(len(gdf), dtype=int)
strt = np.full(len(gdf), 12.345)
ibound[left] = ibound[right] = -1
strt[left], strt[right] = 20.0, 10.0
flopy.mfusg.MfUsgBas(model, ibound=ibound, strt=strt)
flopy.mfusg.MfUsgLpf(model, laytyp=0, hk=conductivity, vka=conductivity, ipakcb=53)
flopy.mfusg.MfUsgSms(
    model, nonlinmeth=0, linmeth=2, clin="cg",  # Symmetric confined flow
    hclose=1e-8, hiclose=1e-8, rclosepcgu=1e-8, mxiter=100, iter1=500,
)
flopy.modflow.ModflowOc(
    model, stress_period_data={(0, 0): ["save head", "save budget", "print budget"]},
    compact=True,
)

## Write, run and check

Require normal termination, finite heads within the two boundary values, and absolute water-budget discrepancy below 0.1%. Also check the physical solution: `h(x) = 20 - 10 x / 200` m, and Darcy flow `Q = K × width × thickness × (20 - 10) / length = 500` m³/day. A balanced water budget alone cannot establish the correct flow magnitude.

The head tolerance is 10⁻⁵ m and the relative flow tolerance is 10⁻⁵ for this simple, unmodified Voronoi grid. The simulation does not read GSF; that companion supplies geometry for visualization and external import.

In [ ]:
model.write_input()
assert (model_ws / "model.disu").is_file() and gsf_path.is_file()
success, output = model.run_model(silent=True, report=True)
assert success, "MODFLOW-USG failed:\n" + "\n".join(output)
assert any("normal termination" in line.lower() for line in output), "No normal termination."

head_file = flopy.utils.HeadUFile(model_ws / "model.hds")
assert np.isclose(head_file.get_times()[-1], 1.0), "Unexpected output time."
heads = head_file.get_data()[0].ravel()
head_file.close()
assert len(heads) == len(gdf) and np.isfinite(heads).all()
assert heads.min() >= 10.0 - 1e-5 and heads.max() <= 20.0 + 1e-5
exact_heads = 20.0 - 10.0 * x / 200.0
np.testing.assert_allclose(heads, exact_heads, atol=1e-5, rtol=0)

budget = flopy.utils.MfusgListBudget(model_ws / "model.list")
incremental, _ = budget.get_budget()
assert incremental is not None and len(incremental), "No water budget found."
discrepancy = np.asarray(incremental["PERCENT_DISCREPANCY"])
assert np.isfinite(discrepancy).all() and np.abs(discrepancy).max() < 0.1
expected_flow = conductivity * 100.0 * (top - botm[0]) * (20.0 - 10.0) / 200.0
inflow = float(incremental["CONSTANT_HEAD_IN"][-1])
outflow = float(incremental["CONSTANT_HEAD_OUT"][-1])
np.testing.assert_allclose([inflow, outflow], expected_flow, rtol=1e-5, atol=0)
print(f"{len(gdf)} cells; maximum head error {np.max(np.abs(heads - exact_heads)):.3g} m")
print(f"Inflow {inflow:g}; outflow {outflow:g}; analytical flow {expected_flow:g} m³/day")
print(f"Maximum budget discrepancy {np.abs(discrepancy).max():.6g}%")

## Plot heads on the original polygons

Assign heads through the map's explicit horizontal row positions and model-node numbers. This remains traceable if the input index or generator IDs are not consecutive.

In [ ]:
mapped_heads = np.empty(len(gdf))
mapped_heads[layer_map["cell_id"].to_numpy(dtype=int)] = heads[
    layer_map["model_node"].to_numpy(dtype=int)
]
results = gdf.assign(hydraulic_head=mapped_heads)
fig, ax = plt.subplots(figsize=(9, 4))
results.plot(
    column="hydraulic_head", cmap="viridis", edgecolor="0.3", linewidth=0.3,
    legend=True, legend_kwds={"label": "Head (m)"}, ax=ax,
)
ax.set(xlabel="Easting (m)", ylabel="Northing (m)", title="Steady confined flow")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

# Remove the temporary files only after inspecting the results.
workspace.cleanup()